<a href="https://colab.research.google.com/github/subham-28/PyTorch/blob/main/nn_module_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn

In [10]:
class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear1=nn.Linear(num_features,3)
    self.relu=nn.ReLU()

    self.linear2=nn.Linear(3,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out=self.linear1(features)
    out=self.relu(out)
    out=self.linear2(out)
    out=self.sigmoid(out)

    return out

In [11]:
features=torch.rand(10,5)

model=Model(features.shape[1])
model(features) #model.forward(features)

tensor([[0.5274],
        [0.5894],
        [0.5972],
        [0.5753],
        [0.5715],
        [0.5693],
        [0.5003],
        [0.5904],
        [0.5998],
        [0.5823]], grad_fn=<SigmoidBackward0>)

In [13]:
model.linear1.weight

Parameter containing:
tensor([[-0.0353,  0.1052,  0.4365,  0.3922, -0.3947],
        [ 0.1572,  0.4241, -0.3706, -0.1889,  0.1835],
        [-0.2074, -0.2752,  0.3117, -0.0446, -0.1152]], requires_grad=True)

In [14]:
model.linear1.bias

Parameter containing:
tensor([-0.0802, -0.1893, -0.4387], requires_grad=True)

In [8]:
!pip install torchinfo

In [15]:
from torchinfo import summary
summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Use of sequential Container

In [17]:
class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network=nn.Sequential(
      nn.Linear(num_features,3),
      nn.ReLU(),
      nn.Linear(3,1),
      nn.Sigmoid()
    )

  def forward(self,features):
    out=self.network(features)
    return out

In [18]:
features=torch.rand(10,5)

model=Model(features.shape[1])
model(features) #model.forward(features)

tensor([[0.5711],
        [0.5763],
        [0.5477],
        [0.5478],
        [0.5628],
        [0.5685],
        [0.5478],
        [0.5705],
        [0.5477],
        [0.5477]], grad_fn=<SigmoidBackward0>)

## Back to pipelining

In [19]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [20]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [21]:
df.shape

(569, 33)

In [22]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

In [23]:
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=['diagnosis']),df['diagnosis'],test_size=0.2,random_state=42)

In [24]:
#scaling
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [25]:
#label encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [26]:
#numpy arr to pytorch tensors
x_train_tensor=torch.from_numpy(x_train)
x_test_tensor=torch.from_numpy(x_test)

y_train_tensor=torch.from_numpy(y_train)
y_test_tensor=torch.from_numpy(y_test)

In [52]:
# Convert tensors to float32 and adjust shape for y_train_tensor
x_train_tensor = x_train_tensor.float()
y_train_tensor = y_train_tensor.float().unsqueeze(1)
x_test_tensor = x_test_tensor.float()
y_test_tensor = y_test_tensor.float().unsqueeze(1)

### Model Building

In [56]:
import torch.nn as nn

class MyNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)

    return out


In [61]:
learning_rate=0.1
epochs=100

In [62]:
loss_function=nn.BCELoss()

In [63]:
# create model
model=MyNN(x_train_tensor.shape[1])

#define optimizer
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

for epoch in range(epochs):

  #forward pass
  y_pred=model(x_train_tensor)

  #loss
  loss = loss_function(y_pred,y_train_tensor.reshape(-1,1))

  #zero gradient
  optimizer.zero_grad()

  #backward pass
  loss.backward()

  #update params
  optimizer.step()

  #print loss
  print(f'Epoch: {epoch+1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.6729697585105896
Epoch: 2, Loss: 0.5137892365455627
Epoch: 3, Loss: 0.4295251965522766
Epoch: 4, Loss: 0.3775106370449066
Epoch: 5, Loss: 0.34179508686065674
Epoch: 6, Loss: 0.31546682119369507
Epoch: 7, Loss: 0.2950628697872162
Epoch: 8, Loss: 0.27865615487098694
Epoch: 9, Loss: 0.2650870382785797
Epoch: 10, Loss: 0.2536146938800812
Epoch: 11, Loss: 0.24374251067638397
Epoch: 12, Loss: 0.23512445390224457
Epoch: 13, Loss: 0.22751140594482422
Epoch: 14, Loss: 0.22071899473667145
Epoch: 15, Loss: 0.21460740268230438
Epoch: 16, Loss: 0.20906861126422882
Epoch: 17, Loss: 0.2040175050497055
Epoch: 18, Loss: 0.19938595592975616
Epoch: 19, Loss: 0.1951187252998352
Epoch: 20, Loss: 0.19117039442062378
Epoch: 21, Loss: 0.18750318884849548
Epoch: 22, Loss: 0.18408545851707458
Epoch: 23, Loss: 0.18089032173156738
Epoch: 24, Loss: 0.17789487540721893
Epoch: 25, Loss: 0.1750793755054474
Epoch: 26, Loss: 0.1724267303943634
Epoch: 27, Loss: 0.16992202401161194
Epoch: 28, Loss: 0.16

In [64]:
# evaluation
with torch.no_grad():
  y_pred = model(x_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5280086398124695
